In [1]:
import copy
import time
import tracemalloc



**Sudoku**

Sudokus von sudoku.com.
Die 0 repräsentiert ein leeres Feld

'Sudoku' ist ein Sudoku aus der Kategorie leicht mit 44 freien Feldern
'SudokuDiff' ist ein Sudoku aus der Kategorie Extrem mit 59 freien Feldern
Ignorieren wir die Arithmetik wären das etwa 9^59, im Vergleich zu 9^44 Möglichkeiten

In [2]:
sudoku = [
    [0, 0, 6, 0, 0, 0, 8, 0, 2],
    [7, 0, 0, 4, 2, 8, 0, 9, 6],
    [2, 1, 0, 0, 3, 0, 7, 0, 5],

    [0, 3, 1, 0, 0, 0, 9, 8, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 7],
    [8, 2, 0, 9, 5, 0, 0, 0, 3],

    [3, 0, 0, 0, 0, 2, 0, 6, 0],
    [0, 8, 5, 0, 7, 6, 0, 0, 0],
    [9, 0, 2, 5, 0, 1, 3, 0, 8]
]

sudokuDiff = [
    [0, 0, 6, 9, 0, 0, 0, 0, 0],
    [4, 0, 0, 0, 0, 0, 0, 0, 8],
    [0, 0, 0, 0, 5, 0, 0, 7, 0],

    [6, 0, 0, 0, 0, 4, 0, 0, 0],
    [1, 0, 8, 6, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 9, 5, 2, 0],

    [0, 0, 5, 3, 0, 0, 0, 0, 0],
    [0, 9, 0, 0, 0, 0, 8, 0, 0],
    [2, 4, 3, 0, 6, 0, 0, 0, 0]
]

sudoku1 = copy.deepcopy(sudoku)
sudoku2 = copy.deepcopy(sudoku)
sudoku3 = copy.deepcopy(sudoku)

sudokuDiff2 = copy.deepcopy(sudokuDiff)
sudokuDiff3 = copy.deepcopy(sudokuDiff)

#funktion zum schönen ausgeben
def print_sudoku(board):
    for a in range(9):
        if a % 3 == 0 and a != 0:
            print("-" * 21)

        for b in range(9):
            if b % 3 == 0 and b != 0:
                print("|", end=" ")

            print(board[a][b], end=" ")
        print()

print_sudoku(sudoku)

0 0 6 | 0 0 0 | 8 0 2 
7 0 0 | 4 2 8 | 0 9 6 
2 1 0 | 0 3 0 | 7 0 5 
---------------------
0 3 1 | 0 0 0 | 9 8 0 
0 0 0 | 1 0 0 | 0 0 7 
8 2 0 | 9 5 0 | 0 0 3 
---------------------
3 0 0 | 0 0 2 | 0 6 0 
0 8 5 | 0 7 6 | 0 0 0 
9 0 2 | 5 0 1 | 3 0 8 


Hilfsmethoden:

In [3]:
#finde nächstes feld
def find_empty(board):
    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                return (row, col)
    return None

#Überprüfen, ob die Zahl erlaubt ist
def is_valid(board, num, row, col):
    #Zeile prüfen, ob schon vorkommt
    for b in range(9):
        if board[row][b] == num and b != col:
            return False

    #Spalte prüfen
    for a in range(9):
        if board[a][col] == num and a != row:
            return False

    #Block prüfen
    start_row = (row // 3) * 3
    start_col = (col // 3) * 3

    for a in range(start_row, start_row + 3):
        for b in range(start_col, start_col + 3):
            if board[a][b] == num and (a, b) != (row, col):
                return False

    return True



# **Depth First Search Algorhitmus**


**DFS auf einfachem Sudoku**

hier braucht der DFS nur sehr wenig Zeit

In [4]:
def solve_sudoku(board):
    empty = find_empty(board)

    #falls kein leeres => Sudoku gelöst
    if empty is None:
        return True

    row, col = empty

    # Probiere Zahlen von 1 bis 9
    for num in range(1, 10):
        if is_valid(board, num, row, col):
            board[row][col] = num  # Zahl

            if solve_sudoku(board):
                return True

            board[row][col] = 0

    return False
tracemalloc.start()
start = time.perf_counter()
if solve_sudoku(sudoku1):
    print("\n Sudoku:")
    print_sudoku(sudoku1)
else:
    print("Keine Lösung gefunden")
end = time.perf_counter()
print(f"{end - start} s")
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"{peak/1024} KB")


 Sudoku:
4 9 6 | 7 1 5 | 8 3 2 
7 5 3 | 4 2 8 | 1 9 6 
2 1 8 | 6 3 9 | 7 4 5 
---------------------
5 3 1 | 2 6 7 | 9 8 4 
6 4 9 | 1 8 3 | 2 5 7 
8 2 7 | 9 5 4 | 6 1 3 
---------------------
3 7 4 | 8 9 2 | 5 6 1 
1 8 5 | 3 7 6 | 4 2 9 
9 6 2 | 5 4 1 | 3 7 8 
0.005351500120013952 s
26.8115234375 KB


**DFS auf extremem Sudoku**

Hier braucht der DFS deutlich länger, um auf das Ergebnis zu kommen und hat trotzdem nur einen leicht erhöhten Speicherverbrauch, der sich mit der erhöhten Tiefe des Suchbaums wegen der erhöhten Anzahl leerer Felder erklären lässt.

In [5]:
def solve_sudoku(board):
    empty = find_empty(board)

    #falls kein leeres => Sudoku gelöst
    if empty is None:
        return True

    row, col = empty

    #Probiere Zahlen von 1 bis 9
    for num in range(1, 10):
        if is_valid(board, num, row, col):
            board[row][col] = num  #Zahl

            if solve_sudoku(board):
                return True

            board[row][col] = 0

    return False
tracemalloc.start()
start = time.perf_counter()
if solve_sudoku(sudokuDiff):
    print("\n Sudoku:")
    print_sudoku(sudokuDiff)
else:
    print("Keine Lösung gefunden")
end = time.perf_counter()
print(f"{end - start} s")
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"{peak/1024} KB")


 Sudoku:
5 8 6 | 9 4 7 | 3 1 2 
4 3 7 | 2 1 6 | 9 5 8 
9 1 2 | 8 5 3 | 6 7 4 
---------------------
6 2 9 | 5 3 4 | 7 8 1 
1 5 8 | 6 7 2 | 4 3 9 
3 7 4 | 1 8 9 | 5 2 6 
---------------------
8 6 5 | 3 9 1 | 2 4 7 
7 9 1 | 4 2 5 | 8 6 3 
2 4 3 | 7 6 8 | 1 9 5 
20.500117708928883 s
30.89453125 KB


**DFS mit Beschränkung**

Der DFS mit Beschränkung ist bei der Lösung eines Sudokus wenig hilfreich, da die Beschränkung schon im Vorhinein durch die Anzahl der freien Felder(AfF) gegeben ist.

Eine Begrenzung < AfF würde nie zu einem Ergebnis führen, da das Sudoku nicht ausgefüllt wird.

Eine Begrenzung > AfF würde nichts bringen, da die Berechnung des Sudokus nach der AfF bereits beendet ist und es keine weiteren ausfüllbaren Felder gibt

In [6]:
#DFS mit Beschränkung
limit = 59
depth = 0
def solve_sudoku(board, depth):
    empty = find_empty(board)

    #falls kein leeres Feld mehr → Sudoku gelöst
    if empty is None:
        return True
    if depth > limit:
        return False


    row, col = empty


    #Probiere Zahlen von 1 bis 9
    for num in range(1, 10):
        if is_valid(board, num, row, col):
            board[row][col] = num  #Zahl

            if solve_sudoku(board, depth+1):
                return True

            board[row][col] = 0

    return False


tracemalloc.start()
start = time.perf_counter()
if solve_sudoku(sudokuDiff2,0):
    print("\n Sudoku:")
    print_sudoku(sudokuDiff2)
else:
    print("Keine Lösung gefunden")
end = time.perf_counter()
print(f"{end - start} s")
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"{peak/1024} KB")


 Sudoku:
5 8 6 | 9 4 7 | 3 1 2 
4 3 7 | 2 1 6 | 9 5 8 
9 1 2 | 8 5 3 | 6 7 4 
---------------------
6 2 9 | 5 3 4 | 7 8 1 
1 5 8 | 6 7 2 | 4 3 9 
3 7 4 | 1 8 9 | 5 2 6 
---------------------
8 6 5 | 3 9 1 | 2 4 7 
7 9 1 | 4 2 5 | 8 6 3 
2 4 3 | 7 6 8 | 1 9 5 
20.713499208912253 s
31.0107421875 KB


**Iterativer DFS**



In [7]:
#Iterativer DFS
limit = 1
depth = 0
def solve_sudoku(board, depth):
    empty = find_empty(board)

    #Falls kein leeres Feld mehr → Sudoku gelöst
    if empty is None:
        return True
    if depth > limit:
        return False


    row, col = empty


    #Probiere Zahlen von 1 bis 9
    for num in range(1, 10):
        if is_valid(board, num, row, col):
            board[row][col] = num  #Zahl

            if solve_sudoku(board, depth+1):
                return True

            board[row][col] = 0

    return False

tracemalloc.start()
start = time.perf_counter()
while not solve_sudoku(sudoku3, 0):
    limit = limit + 1
end = time.perf_counter()
print_sudoku(sudoku3)
print(f"{end - start} s")
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"{peak/1024} KB")



4 9 6 | 7 1 5 | 8 3 2 
7 5 3 | 4 2 8 | 1 9 6 
2 1 8 | 6 3 9 | 7 4 5 
---------------------
5 3 1 | 2 6 7 | 9 8 4 
6 4 9 | 1 8 3 | 2 5 7 
8 2 7 | 9 5 4 | 6 1 3 
---------------------
3 7 4 | 8 9 2 | 5 6 1 
1 8 5 | 3 7 6 | 4 2 9 
9 6 2 | 5 4 1 | 3 7 8 
0.06425062497146428 s
30.3974609375 KB
